# Project 4 — GPU Benchmark Lab (vector-add, matmul, reduction, softmax)
```
NumPy-CPU  vs  torch-CPU  vs  torch-GPU (auto-skips without CUDA)
vec-add | matmul | reduction | row-softmax   ->   runtime vs size, GB/s, GFLOP/s
```
**Goal:** answer *which ops are memory-bound vs compute-bound, and why a GPU wins some but not all of them* — with numbers.
**Learning objectives:** time GPU code correctly (warmup + sync + median); compute achieved GB/s and GFLOP/s; explain scaling with parallelism + bandwidth; predict when the GPU does NOT win (small sizes, transfer-bound).
Runs on CPU-only boxes: GPU rows are skipped with a message; rerun on DGX Spark to fill them.
Data: synthetic random tensors (no download, fully offline); fixed seeds.


In [ ]:
import os  # need os to create results dir and query cpu count
import platform  # need platform to log machine/cpu info for the report
import random  # need random to seed python rng for reproducibility
import time  # need time for perf_counter high-resolution timing
import csv  # need csv to write the benchmark table portably
import numpy as np  # need numpy as the cpu-baseline compute engine
import matplotlib.pyplot as plt  # need matplotlib to plot runtime/throughput curves
import torch  # need torch for torch-cpu and (if present) torch-gpu backends
SEED = 42  # fix a seed constant so every run generates identical tensors
random.seed(SEED)  # seed python rng (shuffling/sampling determinism)
np.random.seed(SEED)  # seed numpy rng (data generation determinism)
torch.manual_seed(SEED)  # seed torch rng (init + data determinism)
os.makedirs('./results', exist_ok=True)  # create results folder (no error if exists)
HAS_CUDA = torch.cuda.is_available()  # detect nvidia gpu (false on cpu-only boxes)
DEV = torch.device('cuda' if HAS_CUDA else 'cpu')  # pick default torch device once
print('torch', torch.__version__, '| cuda?', HAS_CUDA, '| device:', DEV)  # log env for the report
print('cpu:', platform.processor() or platform.machine(), '| threads:', torch.get_num_threads())  # log cpu + torch threads
if HAS_CUDA:  # only query gpu properties when a gpu is present
    print(torch.cuda.get_device_name(0))  # print gpu model string (e.g. gb10)
    print(f"vram total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")  # print total vram
def sync_if_cuda():  # helper that blocks until queued gpu kernels finish
    if HAS_CUDA:  # only synchronize when cuda exists (noop on cpu boxes)
        torch.cuda.synchronize()  # wait for all async cuda kernels to complete
def time_op(fn, warmup=3, repeat=10):  # generic median timer with warmup + cuda sync
    for _ in range(warmup):  # run warmup iters to warm caches and cuda autotune
        fn()  # execute once without timing (not recorded)
    sync_if_cuda()  # sync before timed region (cuda launches are async)
    ts = []  # collect per-iteration elapsed times here
    for _ in range(repeat):  # run the timed repetitions
        t0 = time.perf_counter()  # start high-resolution wall timer
        fn()  # run the benchmarked op once
        sync_if_cuda()  # sync after so the timer includes gpu work
        ts.append(time.perf_counter() - t0)  # record elapsed seconds for this iter
    return float(np.median(ts))  # return median time (robust to outlier iters)
print('timer ready: warmup=3, repeat=10, median reported')  # confirm harness is defined

In [ ]:
def np_vecadd(a, b):  # numpy vector-add baseline: c = a + b elementwise
    return a + b  # numpy dispatches to vectorized c/blas loops (single thread per chunk)
def torch_vecadd(a, b):  # torch vector-add: same math, torch dispatch (cpu or cuda tensor)
    return a + b  # torch elementwise add kernel on whichever device tensors live
def np_matmul(A, B):  # numpy matmul baseline: C = A @ B via openblas/mkl
    return A @ B  # calls optimized blas dgemm/sgemm under the hood
def torch_matmul(A, B):  # torch matmul: same math via torch backend (mkl/cublas)
    return A @ B  # dispatches to the device/compiler kernel for the tensor type
def np_reduce(x):  # numpy reduction baseline: sum all elements to one scalar
    return x.sum()  # single-pass accumulator over contiguous memory
def torch_reduce(x):  # torch reduction: same sum via torch kernels
    return x.sum()  # parallel tree-reduction on gpu, vectorized loop on cpu
def np_softmax(logits):  # numpy row-softmax baseline with max-subtraction for stability
    m = logits - logits.max(axis=1, keepdims=True)  # subtract row-max to prevent exp overflow
    e = np.exp(m)  # exponentiate shifted logits elementwise
    return e / e.sum(axis=1, keepdims=True)  # normalize each row to sum to 1
def torch_softmax(logits):  # torch row-softmax: same math via torch ops
    m = logits - logits.max(dim=1, keepdim=True).values  # subtract row-max (stable)
    e = torch.exp(m)  # elementwise exp kernel
    return e / e.sum(dim=1, keepdim=True)  # normalize rows to probabilities
rng = np.random.default_rng(SEED)  # create seeded generator for the correctness check
a = rng.standard_normal(1024, dtype=np.float32)  # small test vector a (n=1024)
b = rng.standard_normal(1024, dtype=np.float32)  # small test vector b (same shape)
assert np.allclose(np_vecadd(a, b), (torch_vecadd(torch.from_numpy(a), torch.from_numpy(b))).numpy(), atol=1e-5)  # vec-add must match
A = rng.standard_normal((64, 64), dtype=np.float32)  # small test matrix a (64x64)
B = rng.standard_normal((64, 64), dtype=np.float32)  # small test matrix b (64x64)
assert np.allclose(np_matmul(A, B), (torch_matmul(torch.from_numpy(A), torch.from_numpy(B))).numpy(), atol=1e-4)  # matmul must match
assert np.allclose(np_reduce(a), float(torch_reduce(torch.from_numpy(a))), atol=1e-4)  # reduction must match
L = rng.standard_normal((8, 32), dtype=np.float32)  # small logits block (8 rows x 32 cols)
assert np.allclose(np_softmax(L), (torch_softmax(torch.from_numpy(L))).numpy(), atol=1e-5)  # softmax must match
print('correctness: numpy == torch on vecadd/matmul/reduce/softmax (small tensors)')  # confirm all four ops agree

In [ ]:
rows = []  # collect one dict per (op, backend, size) benchmark point
gen = np.random.default_rng(SEED)  # seeded generator so benchmark tensors are identical each run
VEC_NS = [10_000, 100_000, 1_000_000, 4_000_000]  # vector lengths: 10k (overhead-bound) .. 4m (~16mb, bandwidth-bound)
for n in VEC_NS:  # sweep vector length (problem size) for the memory-bound op
    an = gen.standard_normal(n, dtype=np.float32)  # random numpy vector a on cpu
    bn = gen.standard_normal(n, dtype=np.float32)  # random numpy vector b on cpu
    at = torch.from_numpy(an)  # torch cpu view of a (shares memory, no copy)
    bt = torch.from_numpy(bn)  # torch cpu view of b (shares memory, no copy)
    t = time_op(lambda: np_vecadd(an, bn))  # time numpy vec-add (median of repeats)
    gbs = (3 * n * 4) / t / 1e9  # achieved bandwidth: 2 reads + 1 write x 4 bytes / seconds
    rows.append({'op': 'vecadd', 'backend': 'numpy-cpu', 'n': n, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log numpy row
    print(f"vecadd n={n:>8d} numpy-cpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}")  # live log of numpy point
    t = time_op(lambda: torch_vecadd(at, bt))  # time torch-cpu vec-add on the same data
    gbs = (3 * n * 4) / t / 1e9  # same 12-bytes-per-element bandwidth math
    rows.append({'op': 'vecadd', 'backend': 'torch-cpu', 'n': n, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log torch-cpu row
    print(f"vecadd n={n:>8d} torch-cpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}")  # live log of torch-cpu point
    if HAS_CUDA:  # gpu point only when cuda is available (skips cleanly on cpu boxes)
        ag = torch.from_numpy(an).to('cuda')  # move a to gpu once (outside timer: measures compute, not pcie)
        bg = torch.from_numpy(bn).to('cuda')  # move b to gpu once (outside timer)
        t = time_op(lambda: torch_vecadd(ag, bg))  # time the gpu kernel on resident data
        gbs = (3 * n * 4) / t / 1e9  # same bandwidth math for the gpu kernel
        rows.append({'op': 'vecadd', 'backend': 'torch-gpu', 'n': n, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log gpu row
        print(f"vecadd n={n:>8d} torch-gpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}")  # live log of gpu point
    else:  # explain the skip so the notebook reads well on cpu-only machines
        print(f'vecadd n={n:>8d} torch-gpu SKIPPED (no cuda; rerun on dgx spark)')  # guidance for gpu rerun
RED_NS = [100_000, 1_000_000, 8_000_000]  # reduction lengths: small .. 8m (~32mb streaming read)
for n in RED_NS:  # sweep reduction size (pure streaming-read op)
    xn = gen.standard_normal(n, dtype=np.float32)  # random numpy vector to reduce
    xt = torch.from_numpy(xn)  # torch cpu view sharing the same memory
    t = time_op(lambda: np_reduce(xn))  # time numpy sum (median)
    gbs = (n * 4) / t / 1e9  # bandwidth: 1 read of 4 bytes per element / seconds
    rows.append({'op': 'reduce', 'backend': 'numpy-cpu', 'n': n, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log numpy row
    print(f"reduce n={n:>8d} numpy-cpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}")  # live log numpy
    t = time_op(lambda: torch_reduce(xt))  # time torch-cpu sum
    gbs = (n * 4) / t / 1e9  # same 1-read bandwidth math
    rows.append({'op': 'reduce', 'backend': 'torch-cpu', 'n': n, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log torch-cpu row
    print(f"reduce n={n:>8d} torch-cpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}")  # live log torch-cpu
    if HAS_CUDA:  # gpu reduction only with cuda present
        xg = torch.from_numpy(xn).to('cuda')  # move data to gpu once (outside timer)
        t = time_op(lambda: torch_reduce(xg))  # time gpu tree-reduction kernel
        gbs = (n * 4) / t / 1e9  # same bandwidth math on gpu
        rows.append({'op': 'reduce', 'backend': 'torch-gpu', 'n': n, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log gpu row
        print(f"reduce n={n:>8d} torch-gpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}")  # live log gpu
print('vecadd + reduction sweeps done:', sum(1 for r in rows if r['op'] in ('vecadd', 'reduce')), 'rows')  # confirm count

In [ ]:
MAT_MS = [128, 256, 512, 1024]  # square matmul sizes: 128 (tiny) .. 1024 (2.1 gflop of work)
for m in MAT_MS:  # sweep matrix size for the compute-bound op (flops grow as m^3)
    An = gen.standard_normal((m, m), dtype=np.float32)  # random numpy matrix a (m x m)
    Bn = gen.standard_normal((m, m), dtype=np.float32)  # random numpy matrix b (m x m)
    At = torch.from_numpy(An)  # torch cpu view of a (shared memory)
    Bt = torch.from_numpy(Bn)  # torch cpu view of b (shared memory)
    flops = 2 * m ** 3  # exact flop count for gemm: 2*m^3 (multiply + add per inner step)
    t = time_op(lambda: np_matmul(An, Bn), warmup=2, repeat=5)  # time numpy/blas matmul (fewer reps: it is slow)
    gflops = flops / t / 1e9  # achieved throughput in gigaflops
    rows.append({'op': 'matmul', 'backend': 'numpy-cpu', 'n': m, 'time_s': t, 'gbs': '', 'gflops': gflops})  # log numpy row
    print(f'matmul m={m:>5d} numpy-cpu time={t:8.3f}s gflops={gflops:7.1f}')  # live log numpy
    t = time_op(lambda: torch_matmul(At, Bt), warmup=2, repeat=5)  # time torch-cpu matmul
    gflops = flops / t / 1e9  # same flop math for torch-cpu
    rows.append({'op': 'matmul', 'backend': 'torch-cpu', 'n': m, 'time_s': t, 'gbs': '', 'gflops': gflops})  # log torch-cpu row
    print(f'matmul m={m:>5d} torch-cpu time={t:8.3f}s gflops={gflops:7.1f}')  # live log torch-cpu
    if HAS_CUDA:  # gpu matmul only when cuda is present
        Ag = torch.from_numpy(An).to('cuda')  # move a to gpu once (outside timer)
        Bg = torch.from_numpy(Bn).to('cuda')  # move b to gpu once (outside timer)
        t = time_op(lambda: torch_matmul(Ag, Bg), warmup=2, repeat=5)  # time cublas/cutlass gemm
        gflops = flops / t / 1e9  # same flop math on gpu
        rows.append({'op': 'matmul', 'backend': 'torch-gpu', 'n': m, 'time_s': t, 'gbs': '', 'gflops': gflops})  # log gpu row
        print(f'matmul m={m:>5d} torch-gpu time={t:8.3f}s gflops={gflops:7.1f}')  # live log gpu
SOFTMAX_COLS = [64, 256, 1024, 4096]  # softmax widths with fixed 1024 rows (attention-head-like shapes)
SOFTMAX_ROWS = 1024  # fix row count so total elements = rows x cols
for c in SOFTMAX_COLS:  # sweep softmax width (memory traffic grows linearly in c)
    Ln = gen.standard_normal((SOFTMAX_ROWS, c), dtype=np.float32)  # random logits block on cpu
    Lt = torch.from_numpy(Ln)  # torch cpu view sharing memory
    elems = SOFTMAX_ROWS * c  # total elements touched by the op
    t = time_op(lambda: np_softmax(Ln))  # time numpy row-softmax (max + exp + normalize = ~3 passes)
    gbs = (4 * elems * 4) / t / 1e9  # approx traffic: ~4 passes x 4 bytes per element / seconds
    rows.append({'op': 'softmax', 'backend': 'numpy-cpu', 'n': c, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log numpy row
    print(f'softmax cols={c:>5d} numpy-cpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}')  # live log numpy
    t = time_op(lambda: torch_softmax(Lt))  # time torch-cpu row-softmax
    gbs = (4 * elems * 4) / t / 1e9  # same 4-pass traffic model
    rows.append({'op': 'softmax', 'backend': 'torch-cpu', 'n': c, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log torch row
    print(f'softmax cols={c:>5d} torch-cpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}')  # live log torch-cpu
    if HAS_CUDA:  # gpu softmax only with cuda present
        Lg = torch.from_numpy(Ln).to('cuda')  # move logits to gpu once (outside timer)
        t = time_op(lambda: torch_softmax(Lg))  # time fused-ish gpu softmax kernels
        gbs = (4 * elems * 4) / t / 1e9  # same traffic model on gpu
        rows.append({'op': 'softmax', 'backend': 'torch-gpu', 'n': c, 'time_s': t, 'gbs': gbs, 'gflops': ''})  # log gpu row
        print(f'softmax cols={c:>5d} torch-gpu time={t*1e3:8.2f}ms gbs={gbs:6.1f}')  # live log gpu
print('matmul + softmax sweeps done; total rows:', len(rows))  # confirm benchmark table size

In [ ]:
with open('./results/bench.csv', 'w', newline='') as f:  # open benchmark csv for writing
    w = csv.DictWriter(f, fieldnames=['op', 'backend', 'n', 'time_s', 'gbs', 'gflops'])  # writer with fixed columns
    w.writeheader()  # write the header row first
    w.writerows(rows)  # write one row per (op, backend, size) point
print('saved ./results/bench.csv with', len(rows), 'rows')  # confirm csv path + size
for r in rows:  # print a compact text table so the notebook reads without opening the csv
    extra = f"gbs={r['gbs']:.1f}" if r['gbs'] != '' else f"gflops={r['gflops']:.1f}"  # pick the metric valid for this op
    print(f"{r['op']:>8s} {r['backend']:>10s} n={r['n']:>8d} time={r['time_s']*1e3:9.2f}ms {extra}")  # one line per point
by_op_backend = {}  # group series as (op, backend) -> (xs, ys) for plotting
for r in rows:  # walk every benchmark row once
    key = (r['op'], r['backend'])  # series key identifies one curve
    by_op_backend.setdefault(key, ([], []))  # create empty x/y lists on first sight
    by_op_backend[key][0].append(r['n'])  # x value is the problem size
    by_op_backend[key][1].append(r['time_s'])  # y value is median seconds
fig, ax = plt.subplots(figsize=(8, 4.5))  # one figure for the runtime-vs-size log-log plot
for (op, be), (xs, ys) in sorted(by_op_backend.items()):  # draw one curve per (op, backend)
    ax.plot(xs, ys, marker='o', label=f'{op}/{be}')  # line + markers with legend label
ax.set_xscale('log')  # log x-axis: sizes span orders of magnitude
ax.set_yscale('log')  # log y-axis: times span microseconds to seconds
ax.set(title='Runtime vs problem size (log-log; lower is better)', xlabel='n (log)', ylabel='seconds (log)')  # label axes
ax.legend(fontsize=8)  # show legend mapping curves to op/backend
ax.grid(True, which='both', alpha=0.3)  # faint grid for readability
fig.tight_layout()  # fix overlapping labels
fig.savefig('./results/bench_runtime.png', dpi=120)  # save runtime figure
plt.show()  # display the runtime figure inline
fig2, axes = plt.subplots(1, 2, figsize=(11, 4))  # second figure: two throughput panels side by side
mem_rows = [r for r in rows if r['gbs'] != '']  # keep memory-ops (vecadd/reduce/softmax) for the gb/s panel
seen = {}  # group memory rows by backend for the left panel
for r in mem_rows:  # walk memory-op rows only
    seen.setdefault((r['op'], r['backend']), ([], []))  # init series on first sight
    seen[(r['op'], r['backend'])][0].append(r['n'])  # x is problem size
    seen[(r['op'], r['backend'])][1].append(float(r['gbs']))  # y is achieved gb/s
for (op, be), (xs, ys) in sorted(seen.items()):  # draw one gb/s curve per series
    axes[0].plot(xs, ys, marker='o', label=f'{op}/{be}')  # line + markers
axes[0].set_xscale('log')  # log x for wide size range
axes[0].set(title='Achieved bandwidth (higher is better)', xlabel='n (log)', ylabel='GB/s')  # label left panel
axes[0].legend(fontsize=8)  # legend for left panel
axes[0].grid(True, which='both', alpha=0.3)  # grid for left panel
mm_rows = [r for r in rows if r['gflops'] != '']  # keep matmul rows for the gflop/s panel
seen2 = {}  # group matmul rows by backend for the right panel
for r in mm_rows:  # walk matmul rows only
    seen2.setdefault(r['backend'], ([], []))  # init series per backend
    seen2[r['backend']][0].append(r['n'])  # x is matrix size m
    seen2[r['backend']][1].append(float(r['gflops']))  # y is achieved gflop/s
for be, (xs, ys) in sorted(seen2.items()):  # draw one gflop/s curve per backend
    axes[1].plot(xs, ys, marker='o', label=f'matmul/{be}')  # line + markers
axes[1].set_xscale('log')  # log x for 128..1024 range
axes[1].set(title='Matmul throughput (higher is better)', xlabel='m (log)', ylabel='GFLOP/s')  # label right panel
axes[1].legend(fontsize=8)  # legend for right panel
axes[1].grid(True, which='both', alpha=0.3)  # grid for right panel
fig2.tight_layout()  # fix layout of both panels
fig2.savefig('./results/bench_throughput.png', dpi=120)  # save throughput figure
plt.show()  # display the throughput figure
print('figures saved: bench_runtime.png, bench_throughput.png')  # confirm plot paths

In [ ]:
print('=== Roofline reading: arithmetic intensity (flops per byte) ===')  # header for the analysis section
print('vecadd: 1 flop / 12 bytes = 0.08 flop/byte -> MEMORY-bound (bandwidth decides)')  # vec-add intensity math
print('reduce: 1 flop / 4 bytes  = 0.25 flop/byte -> MEMORY-bound (streaming read)')  # reduction intensity math
print('softmax: ~5 flops / 16 bytes ~= 0.3 flop/byte -> MEMORY-bound (multi-pass)')  # softmax intensity math
print('matmul 1024: 2.1e9 flops / 12.6mb ~= 171 flop/byte -> COMPUTE-bound (flops decide)')  # matmul intensity math
big = max((r for r in rows if r['op'] == 'vecadd' and r['backend'] == 'torch-cpu'), key=lambda r: float(r['gbs']))  # best torch-cpu vecadd row
print(f"best cpu vecadd bandwidth: {float(big['gbs']):.1f} GB/s at n={big['n']} (compare to ~20-50 gb/s ddr5 ceilings)")  # anchor vs hardware
bigm = max((r for r in rows if r['op'] == 'matmul'), key=lambda r: float(r['gflops']))  # best matmul row across backends
print(f"best matmul: {float(bigm['gflops']):.1f} GFLOP/s via {bigm['backend']} at m={bigm['n']}")  # anchor compute ceiling
print('why gpu wins vecadd at large n: thousands of cores hide latency + 10x hbm bandwidth; at small n kernel-launch (~5-20us) dominates -> cpu wins')  # core gpu lesson
print('why gpu wins matmul sooner: O(n^3) flops reuse each byte O(n) times, so extra flops matter immediately')  # compute-bound lesson
small = [r for r in rows if r['op'] == 'vecadd' and r['n'] == VEC_NS[0]]  # pull the smallest-size vecadd rows
for r in small:  # show that at tiny sizes absolute times are microseconds (launch/overhead regime)
    print(f"  small-n evidence: {r['backend']} time={r['time_s']*1e6:.1f}us (overhead, not bandwidth)")  # microsecond evidence
n = 2_000_000  # what-if size: 2m elements (~8mb, comfortably bandwidth-bound on cpu)
xf32 = gen.standard_normal(n, dtype=np.float32)  # fp32 test vector for the dtype what-if
xf64 = xf32.astype(np.float64)  # same values widened to fp64 (2x the traffic, same flops)
t32 = time_op(lambda: np_vecadd(xf32, xf32))  # time fp32 vec-add (12 bytes/element traffic)
t64 = time_op(lambda: np_vecadd(xf64, xf64))  # time fp64 vec-add (24 bytes/element traffic)
print(f'what-if dtype: fp32 {t32*1e3:.2f}ms ({(3*n*4)/t32/1e9:.1f} GB/s) vs fp64 {t64*1e3:.2f}ms ({(3*n*8)/t64/1e9:.1f} GB/s)')  # fp64 moves 2x bytes
print('what-if takeaway: memory-bound ops scale with BYTES moved, so fp64 costs ~2x; compute-bound matmul would barely move (flops identical)')  # lesson
print('cleanup: benchmark tensors freed on next rerun; rerun on dgx spark to append torch-gpu curves to the same csv')  # next steps
print('next: P5 implements these ops as raw cuda kernels; P6 optimizes one up the roofline')  # forward pointer

## What to write in your README (7 questions)
1. What did I build? 2. How does correct GPU timing work (warmup/sync/median)? 3. Best GB/s + GFLOP/s numbers?
4. What surprised you (small-n CPU wins? numpy vs torch gap)? 5. Bottleneck: which ops are memory- vs compute-bound, and how do you know?
6. What if fp64 / small-n / transfer-included? Why? 7. Next: which op would you write a custom kernel for first (see P5)?
**One-liner:** `intensity = flops / bytes` decides everything — low intensity (vecadd/reduce/softmax) chases bandwidth, high intensity (matmul) chases flops; GPUs win when parallelism × bandwidth outweigh launch + transfer costs.
